# FMR — Real-Model Pipeline (Instance A)

Runs the Faithful Medical Reasoning pipeline on **real** open medical VLMs on a Colab GPU, then pushes results back to the `master` branch automatically.

**Before running:** set two Colab secrets (🔑 left sidebar):
- `HF_TOKEN` — Hugging Face read token (MedGemma also needs its license accepted on HF).
- `GH_TOKEN` — GitHub PAT with `contents: read+write` on `fmr-thesis` (for the auto-push).

Then `Runtime → Run all` with a GPU runtime (T4 is enough for 2–3B models).

**Datasets:** VQA-RAD and PathVQA ship images inline and run with zero setup. SLAKE's annotation mirror has no inline images — its cell downloads the SLAKE image archive and passes `--image-root` (run it last).

In [ ]:
# 1. GPU check
import torch, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:600])
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime -> Change runtime type -> GPU"
print("CUDA OK:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Tokens from Colab secrets
import os
try:
    from google.colab import userdata
    for k in ("HF_TOKEN", "GH_TOKEN"):
        try: os.environ[k] = userdata.get(k)
        except Exception: print(f"WARNING: secret {k} not set")
except Exception:
    print("Not in Colab; expecting HF_TOKEN/GH_TOKEN already in env")
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| GH_TOKEN set:", bool(os.environ.get("GH_TOKEN")))

In [ ]:
# 3. Clone the repo (master = Instance A) and install
import os
REPO = "https://github.com/Ankit-blip737/fmr-thesis.git"
if not os.path.exists("fmr-thesis"):
    tok = os.environ.get("GH_TOKEN")
    url = REPO.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO
    !git clone -b master {url} fmr-thesis
%cd fmr-thesis
!git pull --no-edit
!pip -q install -e fmr[real] || pip -q install numpy scipy pyyaml matplotlib scikit-learn transformers datasets accelerate qwen-vl-utils pillow
print("installed")

## 4. Run the pipeline — VQA-RAD (inline images, runs immediately)

`run_real.py` runs baselines → blind test (with the **headline replication verdict**) → full FMR (signals → FS → conformal gate), writes `fmr/outputs/real/<dataset>/`, and commits + pushes with `--push`. `medvlm_r1` = reasoning model; `qwen25_vl_3b` = non-reasoning baseline. Small real test sets → relax α for a feasible gate (see the calibration-power finding in RESULTS_LOG).

In [ ]:
# VQA-RAD (X-ray/CT) — reasoning (MedVLM-R1) vs non-reasoning (Qwen2.5-VL-3B)
!python fmr/scripts/run_real.py     --dataset vqa_rad --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 400 --n-consistency 5 --alpha 0.15 --push

In [ ]:
# PathVQA (pathology — modality breadth). Larger download; still inline images.
!python fmr/scripts/run_real.py     --dataset pathvqa --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 400 --n-consistency 5 --alpha 0.15 --push

## 5. SLAKE (real modality labels + boxes-adjacent) — needs the image archive

The `BoKelvin/SLAKE` mirror is annotations-only. This cell fetches the SLAKE images from the repo's `imgs.zip` and passes `--image-root` so the loader attaches real images. If the archive URL changes, download SLAKE imgs from https://www.med-vqa.com/slake/ and point `--image-root` at the extracted folder.

In [ ]:
# Fetch SLAKE images (best-effort) then run with --image-root
import os
os.makedirs("slake_imgs", exist_ok=True)
!huggingface-cli download BoKelvin/SLAKE imgs.zip --repo-type dataset --local-dir slake_imgs 2>/dev/null || echo "auto-download failed; place SLAKE imgs under slake_imgs/ manually"
!cd slake_imgs && (unzip -oq imgs.zip 2>/dev/null || echo "no imgs.zip")
IMG_ROOT = "slake_imgs" if os.path.exists("slake_imgs") else None
!python fmr/scripts/run_real.py     --dataset slake --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 400 --n-consistency 5 --alpha 0.10 --image-root {IMG_ROOT} --push

## 6. Inspect results
Figures + JSON are under `fmr/outputs/real/<dataset>/` and already pushed to `master`. On next local resume, Instance A pulls these real FS scores and wires them into calibration (replacing the provisional mock scores).

In [ ]:
import json, glob
from IPython.display import Image, display
for f in sorted(glob.glob("fmr/outputs/real/*/blind_test.json")):
    r = json.load(open(f)); rep = r.get("replication", {})
    print(f"
=== {f} ===")
    print("  HEADLINE:", rep.get("note"), "| drift slope:", round(rep.get("drift_slope", float('nan')), 4))
for f in sorted(glob.glob("fmr/outputs/real/*/fmr_results.json")):
    r = json.load(open(f)); v = r.get("validation", {}); ab = r["abstention"]
    print(f"
=== {f} ===")
    print("  per-modality:", r.get("per_modality"))
    print("  AUROC:", {k: round(v[k],3) for k in v if k.startswith("auroc_")})
    for g in ("fs","confidence"):
        print(f"  abstain[{g}] cov={ab[g]['test']['coverage']:.2f} err={ab[g]['test']['retained_error']} AURC={ab[g]['aurc']:.3f}")
for f in sorted(glob.glob("fmr/outputs/real/*/figures/fig1_grounding_drift.png")):
    display(Image(f))